In [ ]:
import torch
import os
import ast
import pandas as pd
import logging
from itertools import permutations
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# [설정 구간]
MODEL_NAME = "Qwen/Qwen3-14B"
SEPARATOR = " "
DATA_DIR = "./sentence_prediction"
RESULT_PATH = os.path.join(DATA_DIR, "results/experiments.csv")
PRED_SAVE_PATH = os.path.join(DATA_DIR, f"results/pred_{MODEL_NAME.replace('/', '_')}.csv")

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def get_best_order_batched(sentences, separator, model, tokenizer):
    """24개 순열을 배치 처리하여 속도 극대화"""
    perms = list(permutations(range(4)))
    texts = [separator.join([sentences[i] for i in p]) for p in perms]

    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(model.device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask, labels=input_ids)
        logits = outputs.logits

        # Shift tokens for Causal LM Loss
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = input_ids[..., 1:].contiguous()
        shift_mask = attention_mask[..., 1:].contiguous()

        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        loss = loss.view(shift_labels.size())

        # 실제 토큰 부분만 평균 Loss 계산
        mean_loss = (loss * shift_mask).sum(dim=1) / shift_mask.sum(dim=1)

    best_idx = torch.argmin(mean_loss).item()
    return list(perms[best_idx])

def main():
    # 모델 로드
    logger.info(f"Loading Model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True
    )
    model.eval()

    # 데이터 로드
    train_path = os.path.join(DATA_DIR, 'train.csv')
    train_df = pd.read_csv(train_path)
    logger.info(f"Data Loaded: {len(train_df)} rows from {train_path}")

    # 추론
    predictions = []
    correct_count = 0

    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc=f"Evaluating {MODEL_NAME}"):
        sentences = [row['sentence_0'], row['sentence_1'], row['sentence_2'], row['sentence_3']]
        answer = [int(row[f'answer_{i}']) for i in range(4)]

        pred = get_best_order_batched(sentences, SEPARATOR, model, tokenizer)
        predictions.append(str(pred))

        if pred == answer:
            correct_count += 1

    acc = correct_count / len(train_df)
    logger.info(f"Accuracy: {acc:.4f} ({correct_count}/{len(train_df)})")

    # 결과 누적 저장 (기존 experiments.csv 파일 업데이트)
    new_result = pd.DataFrame({
        'model': [MODEL_NAME],
        'separator': [repr(SEPARATOR)],
        'accuracy': [acc]
    })

    if os.path.exists(RESULT_PATH):
        existing_df = pd.read_csv(RESULT_PATH)
        final_df = pd.concat([existing_df, new_result], ignore_index=True)
    else:
        final_df = new_result
        os.makedirs(os.path.dirname(RESULT_PATH), exist_ok=True)

    final_df.to_csv(RESULT_PATH, index=False)

    # 상세 예측 결과 저장
    pd.DataFrame({'pred': predictions}).to_csv(PRED_SAVE_PATH, index=False)

    logger.info(f"All processes completed. Results appended to {RESULT_PATH}")

if __name__ == "__main__":
    main()